# T2V-1K+ · A100 training

Same Tina × RPG pipeline as T2V-1K, scaled up: a twice-as-wide ResNet-20 (1.09M weights), a generic body trained on every ImageNet-1k image with feature distillation towards CLIP (for better transfer to never-seen classes), a 2,000-model zoo built in 8 parallel processes, and a larger generator (hidden size 768, as Tina finds that a larger DiT makes the model smarter) whose joint transformer follows Tina and G.pt: one GPT-2 transformer over [time token, class tokens in task order, parameter tokens]. Same 900 / 100 class split and the same 100-task test splits as T2V-1K.

**Runtime → Change runtime type → A100 GPU (High-RAM)**, then run the cell (~2.5–3 h). Results go to Google Drive; rerunning resumes.

In [ ]:
#@title **T2V-1K+ on an A100** (~2.5-3 h): wider ResNet-20, CLIP-distilled body, 2,000-model zoo, Tina-style generator
# Runtime -> Change runtime type -> A100 GPU (High-RAM). Everything is saved to Google Drive; if the session
# stops, run the cell again and it resumes from the last finished step.
OUT = "/content/drive/MyDrive/t2v_imagenet_plus"
ARCH = "resnet20w2"                         # 1.09M weights (T2V-1K: resnet20, 274K)
PER_CLASS, GENERIC_EPOCHS, DISTILL = 1300, 30, 2.0
ZOO_PER_CLASS, N_TRAIN, N_EVAL, SHARDS = 400, 2000, 100, 8
RECIPE = "--epochs 4 --finetune tail:2 --soup 2 --amp --fp16 --bs 256 --lr 0.02"
GEN_STEPS = 12000
VERSION = "t2v_1k_plus"

import json, os, subprocess, sys, time
T_START = time.time()


def sh(cmd):
    print(f"\n>> {cmd}", flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(s in line for s in ("Warning", "Loading weights", "TransformerEncoder", "warnings.warn")):
            print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"failed after {time.time() - t0:.0f}s: {cmd}")
    print(f"<< done in {time.time() - t0:.0f}s  (total {(time.time() - T_START) / 60:.0f} min)", flush=True)


def parallel(cmds, logs, label):
    procs = [subprocess.Popen(f"{c} > {l} 2>&1", shell=True) for c, l in zip(cmds, logs)]
    print(f"\n{label}: {len(procs)} processes; progress every 2 minutes", flush=True)
    while any(p.poll() is None for p in procs):
        time.sleep(120)
        lines = []
        for l in logs:
            try:
                lines.append(open(l).read().strip().splitlines()[-1].strip()[:60])
            except (OSError, IndexError):
                pass
        print(time.strftime("%H:%M"), f"({(time.time() - T_START) / 60:.0f} min)", " | ".join(lines), flush=True)
    if any(p.returncode for p in procs):
        for l in logs:
            print(f"--- {l}\n" + "".join(open(l).readlines()[-15:]))
        raise RuntimeError(f"{label} failed, see the logs above")


# ---------------------------------------------------------------- setup
REPO = "/content/t2v-text-to-vision"
if not os.path.isdir(REPO):
    sh(f"git clone -q --depth 1 https://github.com/kalidolinux-sudo/t2v-text-to-vision {REPO}")
else:
    sh(f"git -C {REPO} pull -q")
os.chdir(REPO)
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
sh("pip -q install pyarrow")
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)
ZOO, CKPT, py = f"{OUT}/zoo", f"{OUT}/{VERSION}.pt", sys.executable

# ImageNet-1k 32x32, every training image (~1.28M), cached once; the zoo reuses a subset of the same cache
sh(f"{py} -c \"from t2v.data import ImageNet32; d = ImageNet32('data', 'cpu', {PER_CLASS}); "
   f"print(len(d.train_y), 'train /', len(d.test_y), 'test images')\"")

# ---------------------------------------------------------------- 1. generic model, CLIP-distilled
if not os.path.exists(f"{ZOO}/generic.pt"):
    sh(f"{py} -m t2v.zoo generic --dataset imagenet32 --per_class {PER_CLASS} --split_file data/imagenet_split.json "
       f"--arch {ARCH} --out {ZOO} --epochs {GENERIC_EPOCHS} --lr 0.2 --bs 512 --amp --clip_distill {DISTILL} "
       f"--descriptions data/descriptions_imagenet.json")

# ---------------------------------------------------------------- 2. model zoo: 2000 tasks of 2-10 seen classes
parallel([f"{py} -m t2v.zoo pmodels --out {ZOO} --split train --n {N_TRAIN} --min_k 2 {RECIPE} "
          f"--per_class {ZOO_PER_CLASS} --shard {s} --num_shards {SHARDS} --save_every 25" for s in range(SHARDS)],
         [f"{OUT}/zoo_train_{s}.log" for s in range(SHARDS)], "model zoo")

# ---------------------------------------------------------------- 3. test tasks (10 classes each), same recipe
EVAL = {"eval_seen": f"--n_train_ref {N_TRAIN}", "eval_unseen": "", "eval_novel": "", "eval_novel_hard": ""}
parallel([f"{py} -m t2v.zoo pmodels --out {ZOO} --split {s} --n {N_EVAL} {RECIPE} --per_class {ZOO_PER_CLASS} {x} "
          f"--save_every 25" for s, x in EVAL.items()], [f"{OUT}/zoo_{s}.log" for s in EVAL], "test tasks")
sh(f"{py} -m t2v.zoo analyze --out {ZOO} --soup 2")

# ---------------------------------------------------------------- 4. the generator (larger, bf16 on A100)
# joint transformer as in Tina / G.pt: one GPT-2 transformer over [time token, class tokens in order, parameter tokens]
if not os.path.exists(f"{CKPT}.done"):
    sh(f"{py} -m t2v.train_gen --zoo {ZOO} --out {CKPT} --token_size 1024 --d_model 768 --denoiser mlp "
       f"--mlp_hidden 2048 --joint_layers 6 --joint_arch tina --bs 48 --tokens_per_step 8192 "
       f"--steps {GEN_STEPS} --warmup 500 "
       f"--pred x0 --norm element --head_weight 10 --semantic_prior --descriptions data/descriptions_imagenet.json "
       f"--neighbors data/neighbors_imagenet.json --prompt_ensemble --emb_noise 0.02 --ema 0.999 --cosine --bf16 "
       f"--log_every 500 --save_every 1000")
    open(f"{CKPT}.done", "w").write("ok")

# ---------------------------------------------------------------- 5. evaluation (100 tasks per split)
SPLITS = ["eval_seen", "eval_unseen", "eval_novel", "eval_novel_hard"]
out = f"{OUT}/eval_{VERSION}.json"
if not os.path.exists(out):
    sh(f"{py} -m t2v.evaluate --ckpt {CKPT} --zoo {ZOO} --splits {','.join(SPLITS)} --max_tasks {N_EVAL} --samples 1 "
       f"--steps 5 --fp16 --body_noise 0 --gate auto --prior auto --out {out}")
sh(f"{py} -m t2v.diagnose --ckpt {CKPT} --zoo {ZOO} --tasks 50 --steps 5 --fp16 --body_noise 0 "
   f"--splits eval_seen,eval_novel,eval_novel_hard --out {OUT}/diagnose_{VERSION}.json")

# ---------------------------------------------------------------- summary vs T2V-1K
import torch
R = json.load(open(out))
g = torch.load(f"{ZOO}/generic.pt", map_location="cpu", weights_only=False)
OLD = {"eval_seen": (0.700, 0.724), "eval_unseen": (0.641, 0.749), "eval_novel": (0.636, 0.741), "eval_novel_hard": (0.475, 0.648)}
print("\n" + "=" * 92)
print(f"T2V-1K+ ({ARCH}, {R['num_params']:,} weights) vs T2V-1K (resnet20, 274,042) | 100 tasks per split, 10 classes")
print(f"{'':16s}{'T2V-1K+':>10s}{'oracle+':>10s}{'T2V-1K':>10s}{'oracle':>10s}{'gain':>9s}")
for s in SPLITS:
    sm = R[s]["summary"]
    print(f"{s:16s}{sm['generated_mean']:10.3f}{sm['oracle']:10.3f}{OLD[s][0]:10.3f}{OLD[s][1]:10.3f}{sm['generated_mean'] - OLD[s][0]:+9.3f}")
print("distilled body, CLIP-text classifier:", {k: round(g[k], 3) for k in g if k.startswith("zero_shot")})
print("=" * 92)
print(f"final model: {CKPT} ({os.path.getsize(CKPT) / 1e6:.0f} MB); total {(time.time() - T_START) / 60:.0f} min")
